In [ ]:
!pip install lightning

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import lightning as L
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint

# Hyperparameters
latent_dim = 100
batch_size  = 64
lr          = 0.001
epochs      = 100
num_classes = 10
target_digit = 3

# MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_dataset = datasets.MNIST('.', train=True, download=True, transform=transform)
target_idx = (full_dataset.targets == target_digit)
digit3_dataset = torch.utils.data.Subset(full_dataset, target_idx.nonzero().squeeze())
train_loader = DataLoader(digit3_dataset, batch_size=batch_size, shuffle=True)

class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        self.fc = nn.Sequential(
            nn.Linear(latent_dim + num_classes, 256),
            nn.ReLU(True),
            nn.Linear(256, 512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.ReLU(True),
            nn.Linear(1024, 28*28),
            nn.Tanh()
        )

    def forward(self, z, labels):
        y = self.label_emb(labels)
        x = torch.cat([z, y], dim=1)
        return self.fc(x).view(-1, 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28 + num_classes, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x, labels):
        y = self.label_emb(labels)
        x = torch.cat([x.view(x.size(0), -1), y], dim=1)
        return self.fc(x)

class GAN(L.LightningModule):
    def __init__(self):
        super(GAN, self).__init__()
        self.generator     = Generator()
        self.discriminator = Discriminator()
        self.criterion     = nn.BCELoss()
        self.automatic_optimization = False

    def forward(self, z, labels):
        return self.generator(z, labels)

    def configure_optimizers(self):
        optimizer_g = optim.Adam(self.generator.parameters(), lr=lr, betas=(0.5, 0.999))
        optimizer_d = optim.Adam(self.discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
        return [optimizer_g, optimizer_d]

    def training_step(self, batch, batch_idx):
        imgs, _ = batch
        imgs = imgs.view(imgs.size(0), -1)

        # override labels to the single target digit
        labels = torch.full((imgs.size(0),), target_digit, dtype=torch.long, device=self.device)

        real_labels = torch.full((imgs.size(0),), 1, dtype=torch.float, device=self.device)
        fake_labels = torch.full((imgs.size(0),), 0, dtype=torch.float, device=self.device)

        opt1, opt2 = self.optimizers()

        # Train Discriminator
        outputs = self.discriminator(imgs, labels)
        d_loss_real = self.criterion(outputs.squeeze(), real_labels)

        z = torch.randn(imgs.size(0), latent_dim, device=self.device)
        fake_imgs = self.generator(z, labels)
        outputs = self.discriminator(fake_imgs.detach(), labels)
        d_loss_fake = self.criterion(outputs.squeeze(), fake_labels)

        d_loss = d_loss_real + d_loss_fake
        opt2.zero_grad()
        self.manual_backward(d_loss)
        opt2.step()

        # Train Generator
        z = torch.randn(imgs.size(0), latent_dim, device=self.device)
        fake_imgs = self.generator(z, labels)
        outputs = self.discriminator(fake_imgs, labels)
        g_loss = self.criterion(outputs.squeeze(), real_labels)

        opt1.zero_grad()
        self.manual_backward(g_loss)
        opt1.step()

        self.log('d_loss', d_loss, prog_bar=True, on_epoch=True)
        self.log('g_loss', g_loss, prog_bar=True, on_epoch=True)

        return g_loss

    def on_train_epoch_end(self):
        # Generate and plot images at the end of each epoch
        z = torch.randn(64, latent_dim, device=self.device)
        labels = torch.full((64,), target_digit, dtype=torch.long, device=self.device)
        generated_images = self.generator(z, labels).view(64, 28, 28).detach().cpu().numpy()

        plt.figure(figsize=(8, 8))
        for i in range(64):
            plt.subplot(8, 8, i + 1)
            plt.imshow(generated_images[i], cmap='gray')
            plt.axis('off')
        plt.show(block=True)

# Set up TensorBoard logger
tensorboard_logger = TensorBoardLogger("lightning_logs", name="GAN")

# Set up model checkpointing
checkpoint_callback = ModelCheckpoint(
    monitor="g_loss",
    dirpath="checkpoints",
    filename="gan-best-model",
    save_top_k=1,
    mode="min",
    verbose=True
)

# Training the GAN
model = GAN()
trainer = L.Trainer(
    max_epochs=epochs,
    accelerator="auto",
    logger=tensorboard_logger,
    callbacks=[checkpoint_callback],
    log_every_n_steps=1
)
trainer.fit(model, train_loader)
